# Week 13 — Cloud Computing and Spatial Assets

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Cloud Computing · STAC, COG, raster windows

**Learning outcome.** Inspect asset metadata and read a raster window.

**What you will implement:** JSON, STAC metadata, rasterio window reads.  
**Mathematical model:** Data extent, shape, pixel support, byte ranges.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Data extent, shape, pixel support, byte ranges

**Method family:** JSON, STAC metadata, rasterio window reads.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
import json
item=json.loads((DATA/'thermal_june_stac_item.json').read_text())
print('STAC Item:',item['id'])
print('Date:',item['properties']['datetime'])
print('Raster asset:',item['assets']['LST']['href'])
print('BBox lon/lat:',item['bbox'])

### Reading the main code

**STAC.** Parse a JSON Item, inspect ID/date/asset href/extent, and locate the linked teaching raster.

- `json.` — Reads or writes structured GeoJSON / STAC metadata.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
try:
    import rasterio
    from rasterio.windows import Window
    with rasterio.open(DATA/item['assets']['LST']['href']) as src:
        tile=src.read(1,window=Window(4,3,5,5))
        print('CRS:',src.crs,'Grid:',src.width,src.height,'Window shape:',tile.shape)
        plt.imshow(tile,cmap='RdYlBu_r');plt.colorbar(label='Synthetic LST °C');plt.title('Window read: 5 × 5 cells');plt.show()
except ImportError:
    print('Install rasterio in the notebook environment: %pip install rasterio')

### Reading the output code

**Rasterio.** `Window` selects a subset by pixel offsets and dimensions; `src.read(1,window=...)` reads the first band.

- `rasterio` — Reads georeferenced raster bands and metadata.
- `Window` — Restricts raster reading to selected pixel rows and columns.
- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Spatial interpretation.** CRS, nodata and the original measurement support must accompany any window display; raster pixel spacing is not new independent detail.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Change the raster window and confirm source CRS and nodata handling.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.